# Aegis — Phase 1: Model Audit
## `04_model_audit.ipynb`

**Status:** AWAITING MANUAL EXECUTION
**Model:** `model_HistGradientBoosting_Tuned.pkl` (Experiment V1 Baseline)
**Target:** `TARGET_next_purchase_30d`
**Dataset:** Frozen temporal dataset — Online Retail II

---

### Audit Objectives

This notebook audits the V1 next-purchase baseline model before it is reviewed for production consideration.

It answers five questions using evidence only — it does not make final policy decisions:

1. Which features actually drive the model's predictions?
2. Are the predicted probabilities well-calibrated for business use?
3. At what probability threshold does the model best serve Aegis's targeting use case?
4. Where does the model fail, and what types of customers does it misclassify?
5. Is performance stable across customer value and activity cohorts?

---

### Critical Methodology Rules

**VALIDATION SET** — used for:
- Classification threshold selection and operating-point analysis
- Propensity band boundary discovery
- All "what should we use in production?" decisions

**TEST SET** — used for:
- Final unbiased model performance evaluation
- Calibration assessment
- Lift and cohort stability reporting
- Confirming that VAL-derived parameters hold out-of-sample

**TEST is evaluated exactly once, after VAL-based operating parameters are chosen.**

---

### Three Distinct Concepts (Never Conflate)

```
Probability Band         ≠  Classification Threshold  ≠  Campaign Targeting Cutoff
(business communication)    (model decision boundary)    (budget-constrained selection)
```

- **Probability band:** How we communicate a customer's likelihood to the business (e.g., High / Low)
- **Classification threshold:** The scalar cutoff that converts a probability into a binary prediction
- **Campaign targeting cutoff:** Top-N% selection regardless of any threshold — used for constrained budgets

---

### Rules

- DO NOT retrain the model.
- DO NOT modify production application code.
- DO NOT move any artifact to `backend/models/`.
- All threshold/band recommendations start as `PENDING_REVIEW` and are set empirically from validation data.
- This notebook concludes with `AUDIT COMPLETE — AWAITING REVIEW`. The human reviews the generated report before Phase 2 begins.


## Section 2 — Environment & Paths

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import json
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score,
    f1_score, brier_score_loss, log_loss, precision_recall_curve, roc_curve
)
from sklearn.calibration import calibration_curve

EXP_DIR    = Path(".")
MODEL_PATH = EXP_DIR / "model_HistGradientBoosting_Tuned.pkl"
DATA_DIR   = Path("../..").resolve() / "prepared/online_retail_ii/temporal"

TARGET      = "TARGET_next_purchase_30d"
RANDOM_STATE = 42

print(f"Experiment dir  : {EXP_DIR.resolve()}")
print(f"Data dir        : {DATA_DIR}")
print(f"Model path      : {MODEL_PATH.resolve()}")
print(f"Model exists    : {MODEL_PATH.exists()}")


## Section 3 — Load Model & Splits

In [ ]:
# Load frozen model
model = joblib.load(MODEL_PATH)
print(f"Model type: {type(model).__name__}")

# Load data
df     = pd.read_parquet(DATA_DIR / "temporal_dataset.parquet", engine="fastparquet")
splits = pd.read_parquet(DATA_DIR / "dataset_splits.parquet", engine='fastparquet')
df     = df.merge(splits[['customer_id', 'observation_date', 'split']],
                  on=['customer_id', 'observation_date', 'split'], how='inner')

features = [c for c in df.columns if c.startswith('FEATURE_')]

# Leakage assertions
assert not any(c.startswith('TARGET_') for c in features), "Target leaked into features"
assert 'customer_id' not in features
assert 'observation_date' not in features
assert 'split' not in features

# ── VALIDATION SET — threshold / band discovery ──────────────────────────────
df_val   = df[df['split'] == 'VALIDATION'].copy()
X_val    = df_val[features]
y_val    = df_val[TARGET].astype(int)
y_val_prob = model.predict_proba(X_val)[:, 1]

# ── TEST SET — final evaluation only ────────────────────────────────────────
df_test  = df[df['split'] == 'TEST'].copy()
X_test   = df_test[features]
y_test   = df_test[TARGET].astype(int)
y_test_prob = model.predict_proba(X_test)[:, 1]

print(f"\nSplit sizes")
print(f"  VALIDATION : {len(y_val):>7,}  |  positive rate: {y_val.mean():.4f}")
print(f"  TEST       : {len(y_test):>7,}  |  positive rate: {y_test.mean():.4f}")
print(f"\nFeature count: {len(features)}")


## Section 4 — Feature Importance

In [ ]:
from sklearn.inspection import permutation_importance

estimator = model.named_steps['classifier'] if hasattr(model, 'named_steps') else model

# HistGradientBoosting doesn't have feature_importances_, use permutation importance on TEST set
print("Computing permutation importance on TEST set... This may take a moment.")
r = permutation_importance(estimator, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1, scoring='average_precision')
importances = r.importances_mean

feat_imp = (
    pd.DataFrame({'feature': features, 'importance': importances})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

print("── Feature Importance (Permutation on TEST - PR-AUC) ─────────────")
print(feat_imp.to_string(index=False))
print(f"\nTop 5 sum  : {feat_imp.head(5)['importance'].sum():.4f} PR-AUC drop")

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(feat_imp['feature'][::-1], feat_imp['importance'][::-1], color='#4c8cbf')
ax.axvline(feat_imp['importance'].mean(), color='#e05050', linestyle='--', label='Mean drop')
ax.set_xlabel('Feature Importance (PR-AUC Drop)')
ax.set_title('Permutation Feature Importance — V1 Baseline')
ax.legend(); ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(EXP_DIR / 'feature_importance_plot.png', dpi=150)
plt.show()
print("Saved: feature_importance_plot.png")

feat_imp.to_parquet(EXP_DIR / 'feature_importance.parquet', index=False)

## Section 5 — Calibration Evaluation (TEST Set)

Calibration is evaluated on TEST. It is descriptive only — it does not
modify the model or influence threshold selection.


In [ ]:
prob_true, prob_pred = calibration_curve(y_test, y_test_prob, n_bins=10, strategy='quantile')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(prob_pred, prob_true, 'o-', color='#4c8cbf', label='Model (TEST)')
ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
ax.fill_between(prob_pred, prob_true, prob_pred,
                alpha=0.15, color='#e05050', label='Calibration gap')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curve — TEST Set')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.hist(y_test_prob[y_test == 0], bins=40, alpha=0.6, color='#4c8cbf',
        label='Negative (no purchase)', density=True)
ax.hist(y_test_prob[y_test == 1], bins=40, alpha=0.6, color='#e05050',
        label='Positive (purchase)', density=True)
ax.set_xlabel('Predicted Probability')
ax.set_ylabel('Density')
ax.set_title('Score Distribution by Actual Class — TEST Set')
ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Calibration — V1 Baseline (TEST)', fontsize=12)
plt.tight_layout()
plt.savefig(EXP_DIR / 'calibration_plot.png', dpi=150)
plt.show()

brier   = brier_score_loss(y_test, y_test_prob)
mean_p  = y_test_prob.mean()
actual  = y_test.mean()
print(f"Brier Score           : {brier:.4f}")
print(f"Mean predicted prob   : {mean_p:.4f}")
print(f"Actual positive rate  : {actual:.4f}")
gap = mean_p - actual
print(f"Calibration gap       : {gap:+.4f}  "
      f"({'over-predicts probabilities' if gap > 0.02 else 'under-predicts probabilities' if gap < -0.02 else 'within tolerance'})")


## Section 6 — Classification Threshold Analysis (VALIDATION Set)

Operating threshold discovery uses VALIDATION only.

Evaluated metrics:
- Precision, Recall, F1
- Predicted positive rate (what fraction of customers would be targeted)
- Lift and capture rate at top 1/5/10/15/20/30% by score

**The notebook reports evidence. The human sets the production threshold.**


In [ ]:
# ── 6a. Threshold sweep on VALIDATION ───────────────────────────────────────
thresholds = np.round(np.arange(0.05, 0.80, 0.025), 3)
rows = []
for t in thresholds:
    y_pred_t = (y_val_prob >= t).astype(int)
    rows.append({
        'threshold':             t,
        'precision':             round(precision_score(y_val, y_pred_t, zero_division=0), 4),
        'recall':                round(recall_score(y_val, y_pred_t, zero_division=0), 4),
        'f1':                    round(f1_score(y_val, y_pred_t, zero_division=0), 4),
        'predicted_positive_rate': round(y_pred_t.mean(), 4),
    })

thresh_df = pd.DataFrame(rows)

# ── 6b. Targeting budget analysis — top N% by score on VALIDATION ────────────
base_rate_val = y_val.mean()
df_val_ranked = pd.DataFrame({'true': y_val.values, 'prob': y_val_prob})
df_val_ranked = df_val_ranked.sort_values('prob', ascending=False).reset_index(drop=True)
total_pos_val = df_val_ranked['true'].sum()

budget_rows = []
for pct in [0.01, 0.05, 0.10, 0.15, 0.20, 0.30]:
    k         = max(1, int(len(df_val_ranked) * pct))
    top_k     = df_val_ranked.head(k)           # deterministic top-N
    found     = top_k['true'].sum()
    prec_at_k = found / k
    lift      = prec_at_k / base_rate_val if base_rate_val > 0 else 0
    capture   = found / total_pos_val if total_pos_val > 0 else 0
    budget_rows.append({
        'Budget':              f"Top {pct*100:.0f}%",
        'Customers Targeted':  k,
        'Purchasers Found':    int(found),
        'Precision@Budget':    round(prec_at_k, 4),
        'Lift':                round(lift, 3),
        'Capture Rate':        round(capture, 4),
        'Min Score in Group':  round(float(top_k['prob'].min()), 4),
    })

budget_df = pd.DataFrame(budget_rows)

print("── Threshold Sweep (VALIDATION) ────────────────────────────────")
print(thresh_df.to_string(index=False))
print()
print("── Targeting Budget Analysis (VALIDATION) ──────────────────────")
print(budget_df.to_string(index=False))

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
ax.plot(thresh_df['threshold'], thresh_df['precision'], label='Precision', color='#4c8cbf')
ax.plot(thresh_df['threshold'], thresh_df['recall'],    label='Recall',    color='#e05050')
ax.plot(thresh_df['threshold'], thresh_df['f1'],        label='F1',        color='#50a050')
ax.set_xlabel('Threshold'); ax.set_title('Precision / Recall / F1 vs Threshold (VAL)')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(thresh_df['threshold'], thresh_df['predicted_positive_rate'], color='#9050c0')
ax.axhline(base_rate_val, color='gray', linestyle=':', label=f'Base rate ({base_rate_val:.2%})')
ax.set_xlabel('Threshold'); ax.set_ylabel('Predicted Positive Rate')
ax.set_title('Predicted Positive Rate vs Threshold (VAL)')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[2]
x = range(len(budget_df))
ax.bar(x, budget_df['Lift'], color='#4c8cbf', alpha=0.85)
ax.axhline(1.0, color='gray', linestyle='--', label='Random (1.0x)')
ax.set_xticks(x); ax.set_xticklabels(budget_df['Budget'], rotation=30)
ax.set_ylabel('Lift over Random'); ax.set_title('Lift by Targeting Budget (VAL)')
ax.legend(); ax.grid(alpha=0.3, axis='y')

plt.suptitle('Threshold & Targeting Budget Analysis — VALIDATION Set', fontsize=12)
plt.tight_layout()
plt.savefig(EXP_DIR / 'threshold_analysis_plot.png', dpi=150)
plt.show()


## Section 7 — Empirical Propensity Band Discovery (VALIDATION Set)

Inspect how actual purchase rate varies across score deciles on VALIDATION.
Use this evidence to propose candidate band boundaries.

**Bands are not declared here. The human sets them after reviewing this section.**


In [ ]:
df_bands_val = pd.DataFrame({'prob': y_val_prob, 'actual': y_val.values})
df_bands_val['decile'] = pd.qcut(df_bands_val['prob'], q=10, labels=False, duplicates='drop')

band_stats = df_bands_val.groupby('decile').agg(
    n=('actual', 'count'),
    actual_rate=('actual', 'mean'),
    prob_min=('prob', 'min'),
    prob_max=('prob', 'max'),
    prob_mean=('prob', 'mean')
).reset_index()
band_stats['lift_over_base'] = band_stats['actual_rate'] / y_val.mean()

print("── Score Decile Analysis (VALIDATION) ─────────────────────────")
print(band_stats.round(4).to_string(index=False))
print()
print(f"Base purchase rate (VAL): {y_val.mean():.4f}")
print()
print("── Candidate Band Boundaries — TO BE DECIDED AFTER REVIEWING ABOVE ──")
print("  Look for natural breakpoints in the actual_rate column.")
print("  The band boundaries should align with meaningful changes in actual_rate.")
print("  DO NOT use the default 0–20/20–50/50–75/75–100 split without verification.")
print()
print("  Hint: bands that produce lift < 1.0 in the lower deciles may need to be")
print("  merged; bands with very similar actual_rate may not warrant separation.")

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(band_stats['decile'], band_stats['actual_rate'],
       color='#4c8cbf', alpha=0.7, label='Actual purchase rate')
ax.plot(band_stats['decile'], band_stats['prob_mean'],
        'o--', color='#e05050', label='Mean predicted probability')
ax.axhline(y_val.mean(), color='gray', linestyle=':', label=f'Base rate ({y_val.mean():.2%})')
ax.set_xlabel('Score Decile (0 = lowest predicted probability)')
ax.set_ylabel('Rate')
ax.set_title('Actual Purchase Rate vs Mean Predicted Probability by Decile — VALIDATION')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(EXP_DIR / 'propensity_bands_plot.png', dpi=150)
plt.show()


## Section 8 — Operating Parameters (PENDING REVIEW)

**Fill in these values AFTER reviewing Sections 6 and 7.**

Until then they remain `PENDING_REVIEW`.

The notebook will still execute and generate the report with these as placeholders.
The final audit report will clearly mark these as pending.


In [ ]:
# ── FILL IN AFTER REVIEWING SECTIONS 6 & 7 ──────────────────────────────────
# Leave as PENDING_REVIEW until you have reviewed the threshold sweep and
# decile analysis above. Do not copy the default sklearn threshold of 0.5
# without verifying it against the empirical evidence.

OPERATING_THRESHOLD = "PENDING_REVIEW"   # e.g. 0.35 — set after Section 6
TARGETING_CUTOFF_PCT = "PENDING_REVIEW"  # e.g. 0.10 — set after Section 6 budget table

# Propensity band boundaries — set after reviewing Section 7 decile table
# Format: {'band_name': {'min': float, 'max': float, 'rationale': str}}
PROPENSITY_BANDS = "PENDING_REVIEW"
# Example once filled in:
# PROPENSITY_BANDS = {
#     'Very High': {'min': 0.72, 'max': 1.00, 'rationale': 'Deciles 9-10: 3x+ lift'},
#     'High':      {'min': 0.48, 'max': 0.72, 'rationale': 'Deciles 7-8: 1.5-3x lift'},
#     'Moderate':  {'min': 0.22, 'max': 0.48, 'rationale': 'Deciles 4-6: near-base-rate'},
#     'Low':       {'min': 0.00, 'max': 0.22, 'rationale': 'Deciles 1-3: below base rate'},
# }

print("Operating parameters status:")
print(f"  Classification threshold : {OPERATING_THRESHOLD}")
print(f"  Targeting cutoff (top N%): {TARGETING_CUTOFF_PCT}")
print(f"  Propensity bands         : {PROPENSITY_BANDS}")
print()
print("Review Sections 6 and 7 before filling these in.")


## Section 9 — Final TEST Set Evaluation

Evaluate the model on TEST using:
1. Default threshold (0.5) for comparison baseline
2. VAL-derived threshold (if set in Section 8)

TEST is used for descriptive reporting only — no further parameter selection.


In [ ]:
print("── TEST Set — Default Threshold (0.5) ──────────────────────────")
y_pred_default = (y_test_prob >= 0.50).astype(int)
print(f"  ROC-AUC   : {roc_auc_score(y_test, y_test_prob):.4f}")
print(f"  PR-AUC    : {average_precision_score(y_test, y_test_prob):.4f}")
print(f"  Precision : {precision_score(y_test, y_pred_default, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(y_test, y_pred_default, zero_division=0):.4f}")
print(f"  F1        : {f1_score(y_test, y_pred_default, zero_division=0):.4f}")
print(f"  Brier     : {brier_score_loss(y_test, y_test_prob):.4f}")
print(f"  Log Loss  : {log_loss(y_test, y_test_prob):.4f}")

# TEST targeting budget (evaluated descriptively)
base_rate_test = y_test.mean()
df_test_ranked = pd.DataFrame({'true': y_test.values, 'prob': y_test_prob})
df_test_ranked = df_test_ranked.sort_values('prob', ascending=False).reset_index(drop=True)
total_pos_test = df_test_ranked['true'].sum()

test_budget_rows = []
for pct in [0.01, 0.05, 0.10, 0.15, 0.20, 0.30]:
    k         = max(1, int(len(df_test_ranked) * pct))
    top_k     = df_test_ranked.head(k)
    found     = top_k['true'].sum()
    prec_at_k = found / k
    lift      = prec_at_k / base_rate_test if base_rate_test > 0 else 0
    capture   = found / total_pos_test if total_pos_test > 0 else 0
    test_budget_rows.append({
        'Budget':             f"Top {pct*100:.0f}%",
        'Customers':          k,
        'Purchasers Found':   int(found),
        'Precision@Budget':   round(prec_at_k, 4),
        'Lift':               round(lift, 3),
        'Capture Rate':       round(capture, 4),
    })

test_budget_df = pd.DataFrame(test_budget_rows)
print()
print("── TEST Set — Targeting Budget Analysis ────────────────────────")
print(test_budget_df.to_string(index=False))

# If VAL threshold was set, also evaluate it on TEST
if OPERATING_THRESHOLD != "PENDING_REVIEW":
    t = float(OPERATING_THRESHOLD)
    y_pred_val_t = (y_test_prob >= t).astype(int)
    print(f"\n── TEST Set — VAL-derived Threshold ({t}) ───────────────────")
    print(f"  Precision : {precision_score(y_test, y_pred_val_t, zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_test, y_pred_val_t, zero_division=0):.4f}")
    print(f"  F1        : {f1_score(y_test, y_pred_val_t, zero_division=0):.4f}")
    print(f"  Predicted positive rate : {y_pred_val_t.mean():.4f}")
else:
    print("\n  VAL-derived threshold not yet set. Showing default 0.5 only.")


## Section 10 — Error Cohort Analysis (TEST Set)

Profile the four prediction outcome groups using the default 0.5 threshold.
Intended to reveal systematic model failure patterns.

Questions:
- Which customer types does the model consistently miss (False Negatives)?
- Which customer types is it incorrectly targeting (False Positives)?
- Are high-value customers disproportionately in the FN group?


In [ ]:
df_err = df_test.copy()
df_err['prob'] = y_test_prob
df_err['pred'] = (y_test_prob >= 0.50).astype(int)
df_err['true'] = y_test.values

df_err['error_type'] = 'True Negative'
df_err.loc[(df_err['true'] == 1) & (df_err['pred'] == 1), 'error_type'] = 'True Positive'
df_err.loc[(df_err['true'] == 0) & (df_err['pred'] == 1), 'error_type'] = 'False Positive'
df_err.loc[(df_err['true'] == 1) & (df_err['pred'] == 0), 'error_type'] = 'False Negative'

print("── Error Type Counts (default threshold 0.5) ───────────────────")
print(df_err['error_type'].value_counts())

analysis_cols = [
    'FEATURE_recency_days', 'FEATURE_purchase_order_count',
    'FEATURE_total_gross_purchase_spend', 'FEATURE_average_order_value',
    'FEATURE_return_rate', 'FEATURE_purchase_order_count_30d',
    'FEATURE_purchase_order_count_90d', 'FEATURE_is_active_90d', 'prob'
]
available = [c for c in analysis_cols if c in df_err.columns]
cohort_profile = df_err.groupby('error_type')[available].mean().round(4)

print()
print("── Mean Feature Values by Error Type ───────────────────────────")
print(cohort_profile.T.to_string())

# Headline observations
fn = cohort_profile.loc['False Negative'] if 'False Negative' in cohort_profile.index else None
fp = cohort_profile.loc['False Positive'] if 'False Positive' in cohort_profile.index else None
tp = cohort_profile.loc['True Positive']  if 'True Positive'  in cohort_profile.index else None
tn = cohort_profile.loc['True Negative']  if 'True Negative'  in cohort_profile.index else None

print()
print("── Headline Observations ────────────────────────────────────────")
if fn is not None and 'FEATURE_recency_days' in fn.index:
    print(f"  FN avg recency     : {fn['FEATURE_recency_days']:.1f} days")
if fp is not None and 'FEATURE_recency_days' in fp.index:
    print(f"  FP avg recency     : {fp['FEATURE_recency_days']:.1f} days")
if fn is not None and tp is not None and 'FEATURE_total_gross_purchase_spend' in fn.index:
    fn_spend = fn['FEATURE_total_gross_purchase_spend']
    tp_spend = tp['FEATURE_total_gross_purchase_spend']
    ratio    = fn_spend / tp_spend if tp_spend > 0 else float('nan')
    concern  = 'HIGH-VALUE CUSTOMERS BEING MISSED' if ratio > 1.1 else 'comparable to TP'
    print(f"  FN spend / TP spend: {ratio:.2f}x  ({concern})")
if fn is not None and 'FEATURE_return_rate' in fn.index:
    print(f"  FN avg return rate : {fn['FEATURE_return_rate']:.3f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
key_feats = ['FEATURE_recency_days', 'FEATURE_total_gross_purchase_spend', 'FEATURE_return_rate']
titles    = ['Recency (days)', 'Total Gross Spend', 'Return Rate']
colors    = {'True Positive': '#50a050', 'True Negative': '#4c8cbf',
             'False Positive': '#e09020', 'False Negative': '#e05050'}

for i, (feat, title) in enumerate(zip(key_feats, titles)):
    if feat not in df_err.columns:
        continue
    ax = axes[i]
    for et, color in colors.items():
        subset = df_err[df_err['error_type'] == et][feat].dropna()
        if len(subset):
            ax.hist(subset, bins=30, alpha=0.5, label=et, color=color, density=True)
    ax.set_title(title); ax.set_xlabel(feat.replace('FEATURE_', '')); ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.suptitle('Feature Distributions by Prediction Error Type (TEST, default 0.5 threshold)', fontsize=11)
plt.tight_layout()
plt.savefig(EXP_DIR / 'error_cohort_plot.png', dpi=150)
plt.show()


## Section 11 — Cohort Stability Analysis (TEST Set)

Check whether model performance holds across customer value and activity cohorts.
A model performing well on average but poorly on high-value customers
is not safe to deploy.

**Lift@10% uses deterministic top-N selection (not rank percentile).**


In [ ]:
def cohort_metrics(df_sub, prob_col='prob', true_col='true', label=''):
    """Compute ROC-AUC, PR-AUC, Brier and Lift@10% for a cohort subset."""
    y_t = df_sub[true_col].values
    y_p = df_sub[prob_col].values
    n   = len(y_t)
    if len(np.unique(y_t)) < 2 or n < 20:
        return {'cohort': label, 'n': n, 'pos_rate': round(y_t.mean(), 4),
                'note': 'insufficient_positives'}
    # Deterministic top-10% selection
    k      = max(1, int(n * 0.10))
    idx    = np.argsort(y_p)[::-1][:k]          # top-k by score
    top_k  = y_t[idx]
    lift10 = (top_k.mean() / y_t.mean()) if y_t.mean() > 0 else 0

    return {
        'cohort':    label,
        'n':         n,
        'pos_rate':  round(float(y_t.mean()), 4),
        'ROC-AUC':   round(float(roc_auc_score(y_t, y_p)), 4),
        'PR-AUC':    round(float(average_precision_score(y_t, y_p)), 4),
        'Brier':     round(float(brier_score_loss(y_t, y_p)), 4),
        'Lift@10%':  round(float(lift10), 3),
    }

spend_med = df_err['FEATURE_total_gross_purchase_spend'].median()
freq_med  = df_err['FEATURE_purchase_order_count'].median()
rec_q25   = df_err['FEATURE_recency_days'].quantile(0.25)
rec_q75   = df_err['FEATURE_recency_days'].quantile(0.75)

cohorts = [
    ('Full Test Set',      df_err),
    ('High Value',         df_err[df_err['FEATURE_total_gross_purchase_spend'] > spend_med]),
    ('Low Value',          df_err[df_err['FEATURE_total_gross_purchase_spend'] <= spend_med]),
    ('High Frequency',     df_err[df_err['FEATURE_purchase_order_count'] > freq_med]),
    ('Low Frequency',      df_err[df_err['FEATURE_purchase_order_count'] <= freq_med]),
    ('Recently Active',    df_err[df_err['FEATURE_recency_days'] <= rec_q25]),
    ('Moderately Active',  df_err[(df_err['FEATURE_recency_days'] > rec_q25) & (df_err['FEATURE_recency_days'] <= rec_q75)]),
    ('Dormant',            df_err[df_err['FEATURE_recency_days'] > rec_q75]),
]

stability_rows = [cohort_metrics(sub, label=label) for label, sub in cohorts if len(sub) >= 20]
stability_df   = pd.DataFrame(stability_rows)

print("── Cohort Stability — TEST Set ─────────────────────────────────")
print(stability_df.to_string(index=False))

# Flag degraded cohorts
baseline_roc = stability_df.loc[stability_df['cohort'] == 'Full Test Set', 'ROC-AUC']
if len(baseline_roc):
    b = float(baseline_roc.iloc[0])
    print()
    print("── Cohort ROC-AUC vs Full Test Set Baseline ─────────────────────")
    for _, row in stability_df.iterrows():
        if row['cohort'] == 'Full Test Set' or 'note' in row:
            continue
        delta = float(row['ROC-AUC']) - b
        flag  = '  WARNING — significant degradation' if delta < -0.05 else (
                '  OK' if delta > -0.02 else '  Minor degradation')
        print(f"  {row['cohort']:<22}  delta ROC-AUC = {delta:+.4f}  {flag}")


## Section 12 — Audit Report Generation

Generates `model_audit_report.json`.

Operating parameters that are still `PENDING_REVIEW` will be saved as such.
The report status will be `AUDIT_COMPLETE_AWAITING_REVIEW`.

The model is NOT promoted to production.
Phase 2 does NOT begin until this report is reviewed with the team.


In [ ]:
# Threshold evaluation on TEST for report
y_pred_05 = (y_test_prob >= 0.50).astype(int)

test_metrics_default = {
    'threshold_used':  0.50,
    'ROC-AUC':         round(float(roc_auc_score(y_test, y_test_prob)), 4),
    'PR-AUC':          round(float(average_precision_score(y_test, y_test_prob)), 4),
    'Precision':       round(float(precision_score(y_test, y_pred_05, zero_division=0)), 4),
    'Recall':          round(float(recall_score(y_test, y_pred_05, zero_division=0)), 4),
    'F1':              round(float(f1_score(y_test, y_pred_05, zero_division=0)), 4),
    'Brier Score':     round(float(brier_score_loss(y_test, y_test_prob)), 4),
    'Log Loss':        round(float(log_loss(y_test, y_test_prob)), 4),
    'Lift@10%':        float(test_budget_df.loc[test_budget_df['Budget'] == 'Top 10%', 'Lift'].values[0]),
}

audit_report = {
    # Identity
    'model':           str(MODEL_PATH.name),
    'target':          TARGET,
    'feature_count':   len(features),
    'audit_status':    'AUDIT_COMPLETE_AWAITING_REVIEW',
    'audit_completed': pd.Timestamp.now().isoformat(),

    # Data
    'validation_rows':          int(len(y_val)),
    'test_rows':                int(len(y_test)),
    'validation_positive_rate': round(float(y_val.mean()), 4),
    'test_positive_rate':       round(float(y_test.mean()), 4),

    # Model performance (TEST — fixed, unbiased)
    'test_metrics_default_threshold': test_metrics_default,

    # Targeting budget (TEST — descriptive)
    'test_targeting_budget': test_budget_df.to_dict(orient='records'),

    # Calibration (TEST)
    'calibration': {
        'brier_score':             round(float(brier_score_loss(y_test, y_test_prob)), 4),
        'mean_predicted_prob_test': round(float(y_test_prob.mean()), 4),
        'actual_positive_rate_test': round(float(y_test.mean()), 4),
    },

    # Feature importance
    'feature_importance_top5': feat_imp.head(5).to_dict(orient='records'),
    'feature_importance_full': feat_imp.to_dict(orient='records'),

    # Threshold analysis (VALIDATION — discovery)
    'val_threshold_sweep':         thresh_df.to_dict(orient='records'),
    'val_targeting_budget':        budget_df.to_dict(orient='records'),

    # Propensity band discovery (VALIDATION — evidence only)
    'val_score_decile_analysis':   band_stats.round(4).to_dict(orient='records'),

    # Cohort stability (TEST)
    'cohort_stability':            stability_df.to_dict(orient='records'),

    # Error cohort (TEST)
    'error_cohort_profile': (
        df_err.groupby('error_type')[available].mean().round(4).to_dict(orient='index')
    ),

    # Operating parameters — PENDING until human review
    'operating_parameters': {
        'classification_threshold':  OPERATING_THRESHOLD,
        'targeting_cutoff_pct':      TARGETING_CUTOFF_PCT,
        'propensity_bands':          PROPENSITY_BANDS,
        'status':                    'PENDING_REVIEW',
        'note':                      (
            'Review val_threshold_sweep and val_score_decile_analysis, '
            'then set classification_threshold and propensity_bands empirically. '
            'Do not use default sklearn threshold of 0.5 without validation.'
        ),
    },

    # Next step
    'next_step': (
        'Human reviews model_audit_report.json and all plots. '
        'Sets operating_parameters based on empirical evidence. '
        'Approves Phase 2 (Segmentation Discovery) after review.'
    ),
}

with open(EXP_DIR / 'model_audit_report.json', 'w') as f:
    json.dump(audit_report, f, indent=4, default=str)

print("Audit report saved: model_audit_report.json")
print()
print("── Artifacts generated by this notebook ─────────────────────────")
artifacts = [
    'model_audit_report.json        (full audit report)',
    'feature_importance.parquet     (raw importance values)',
    'feature_importance_plot.png',
    'calibration_plot.png',
    'threshold_analysis_plot.png',
    'propensity_bands_plot.png',
    'error_cohort_plot.png',
]
for a in artifacts:
    print(f"  {a}")

print()
print("="*60)
print("  AUDIT COMPLETE — AWAITING REVIEW")
print("="*60)
print()
print("  Operating parameters: PENDING_REVIEW")
print("  Model status        : V1 EXPERIMENTAL BASELINE (unchanged)")
print("  Next action         : Human reviews report + plots.")
print("  Phase 2 start       : After review and approval only.")
print()
print("  DO NOT promote model to backend/models/ before Phase 5.")
